In [167]:
import pandas as pd
from sklearn.neighbors import NearestNeighbors as KNN
from sklearn.preprocessing import MinMaxScaler
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

data_spotify = pd.read_csv('spotify_songs.csv')
data_spotify.drop_duplicates(subset=['track_name', 'track_artist'], inplace=True)
data_spotify.head()

,track_id,track_name,track_artist,track_popularity,track_album_id,track_album_name,track_album_release_date,playlist_name,playlist_id,playlist_genre,playlist_subgenre,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms
0,6f807x0ima9a1j3VPbc7VN,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,66,2oCs0DGTsRO98Gh5ZSl2Cx,I Don't Care (with Justin Bieber) [Loud Luxury...,2019-06-14,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.748,0.916,6,-2.634,1,0.0583,0.1020,0.000000,0.0653,0.518,122.036,194754
1,0r7CVbZTWZgbTCYdfa2P31,Memories - Dillon Francis Remix,Maroon 5,67,63rPSO264uRjW1X5E6cWv6,Memories (Dillon Francis Remix),2019-12-13,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.726,0.815,11,-4.969,1,0.0373,0.0724,0.004210,0.3570,0.693,99.972,162600
2,1z1Hg7Vb0AhHDiEmnDE79l,All the Time - Don Diablo Remix,Zara Larsson,70,1HoSmj2eLcsrR0vE9gThr4,All the Time (Don Diablo Remix),2019-07-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.675,0.931,1,-3.432,0,0.0742,0.0794,0.000023,0.1100,0.613,124.008,176616
3,75FpbthrwQmzHlBJLuGdC7,Call You Mine - Keanu Silva Remix,The Chainsmokers,60,1nqYsOef1yKKuGOVchbsk6,Call You Mine - The Remixes,2019-07-19,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.718,0.930,7,-3.778,1,0.1020,0.0287,0.000009,0.2040,0.277,121.956,169093
4,1e8PAfcKUYoKkxPhrHqw4x,Someone You Loved - Future Humans Remix,Lewis Capaldi,69,7m7vv9wlQ4i0LFuJiE2zsQ,Someone You Loved (Future Humans Remix),2019-03-05,Pop Remix,37i9dQZF1DXcZDD7cfEKhW,pop,dance pop,0.650,0.833,1,-4.672,1,0.0359,0.0803,0.000000,0.0833,0.725,123.976,189052


In [150]:
song_title = 'tough love'

In [165]:
def search_songs(user_title, top_n=1):
    vectorizer = TfidfVectorizer()
    
    titles = data_spotify['track_name'].str.lower()
    titles = titles.dropna()
    
    vectorized_titles = vectorizer.fit_transform(titles)
    user_title_vectorized = vectorizer.transform([user_title.lower()])
    
    similarities = cosine_similarity(user_title_vectorized, vectorized_titles).flatten()
    top_indices = np.argsort(similarities)[::-1][:top_n]
    
    return data_spotify.iloc[top_indices][['track_name', 'track_artist']]
similar = search_songs(song_title)
similar_title = similar['track_name'].iloc[0]
similar

,track_name,track_artist
8,Tough Love - Tiësto Remix / Radio Edit,Avicii


In [169]:
def recommend_songs(liked_song=similar_title, n_recommendations=5):
    scaler = MinMaxScaler()
    imputer = SimpleImputer(strategy='most_frequent')

    num_cols = [name for name, values in data_spotify.items() if (values.dtype == int) | (values.dtype ==                                                                                                float)]

    X_scaled = scaler.fit_transform(data_spotify[num_cols])

    X_num = pd.DataFrame(X_scaled, columns=num_cols)

    genre_dummies = pd.get_dummies(data_spotify['playlist_genre'], dtype=int)
    subgenre_dummies = pd.get_dummies(data_spotify['playlist_subgenre'], dtype=int)

    genre_dummies_df = pd.DataFrame(imputer.fit_transform(genre_dummies), columns = genre_dummies.columns)
    subgenre_dummies_df = pd.DataFrame(imputer.fit_transform(subgenre_dummies), columns =                       subgenre_dummies.columns)

    X = pd.concat([X_num, genre_dummies_df, subgenre_dummies_df], axis=1)

    model = KNN(metric='cosine')
    model.fit(X)

    def find_id(name):
        track_index = data_spotify[data_spotify['track_name'] == name].index
        return track_index

    song_index = find_id(liked_song)

    distance, indices = model.kneighbors(X.iloc[song_index], n_neighbors=(n_recommendations + 1))
    indices = indices[0, :]
    results = data_spotify.iloc[indices]
    results = results[results['track_name'] != liked_song]
    results_titles = results[['track_name', 'track_artist', 'track_album_name']]
    return results_titles
recommend_songs()

,track_name,track_artist,track_album_name
177,Call You Mine - Asketa & Natan Chaim Remix,The Chainsmokers,Call You Mine - The Remixes
243,High Hopes - Don Diablo Remix,Panic! At The Disco,High Hopes (Don Diablo Remix)
308,Think About You - Galantis Remix,Kygo,Think About You (Galantis Remix)
500,I Could Get Used To This,Becky Hill,I Could Get Used To This
495,SHA LA LA,PENTAGON,Genie:us
